In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

link_df = "https://raw.githubusercontent.com/Ptit-Lulu/Projet2_Cinema_Creuse/donnees_df/data_frame_finale2.csv" 

df = pd.read_csv(link_df,sep=';')

In [2]:
df

,tconst,title,startYear,runtimeMinutes,spoken_languages,primaryName,primaryProfession,knownForTitles,averageRating,popularity,tagline,overview,poster_path,video,budget,revenue,genre_1,genre_2,genre_3
0,tt0035423,Kate et Léopold,2001,118,"['en', 'fr', 'it']",James Mangold,"producer,director,writer","tt3315342,tt11563598,tt1950186,tt0358273",6.4,15.770,"If they lived in the same century, they'd be p...",When her scientist ex-boyfriend discovers a po...,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg,False,48000000,76019048,Comedy,Fantasy,Romance
1,tt0052614,Le bel âge,1960,100,['fr'],Pierre Kast,"director,writer,assistant_director","tt0052614,tt0057635,tt0208040,tt0158774",6.8,1.400,Indisponible,"Steph, Jean-Claude and Jacques work in a Paris...",/98Z5yPoNIm8sQeyep5cpr5NHov9.jpg,False,0,0,Comedy,Drama,NaN
2,tt0052686,Certains l'aiment... froide!,1960,90,['fr'],Jean Bastia,"assistant_director,director,writer","tt0315099,tt0054394,tt0052686,tt0159605",5.2,1.916,Indisponible,The old Valmorin died 200 years ago. The notar...,/nWxNWddjRO14xGOiHU770oDfXsV.jpg,False,0,0,Comedy,NaN,NaN
3,tt0052698,Classe tous risques,1960,103,"['fr', 'it']",Claude Sautet,"writer,director,assistant_director","tt0105682,tt0113947,tt0064165,tt0075975",7.5,5.843,Tough Gangster Action,"On crowded Milan streets, two men execute a sp...",/uMiuUAlPwiWDKSgu4SoShpzvMR.jpg,False,0,0,Crime,Drama,Romance
4,tt0052737,Le dialogue des Carmélites,1960,112,['fr'],Inconnu,Inconnu,Inconnu,7.0,2.701,Indisponible,This drama about the Carmelite order of nuns i...,/5yW8duwP8vT0ucs8HtqtLY5neFW.jpg,False,0,0,Drama,History,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7980,tt9845110,Deux,2019,99,"['de', 'fr']",Filippo Meneghetti,"writer,director,assistant_director","tt9845110,tt3454720,tt2447032,tt1054112",7.2,10.742,Indisponible,Nina and Madeleine have been sharing their liv...,/itPHi7V84mUOYuvD3t0QK2oWfVP.jpg,False,0,208723,Drama,Romance,NaN
7981,tt9850344,Police,2020,98,['fr'],Anne Fontaine,"actress,writer,director","tt4370784,tt1035736,tt5989394,tt0119773",5.4,10.722,Indisponible,Three officers are tasked with escorting an il...,/52KQaRQiEIZ9TE4P5iIrRIK7Wej.jpg,False,0,0,Crime,Drama,Thriller
7982,tt9861204,The Love Europe Project,2019,99,"['hr', 'cs', 'en', 'fr', 'de', 'it', 'no', 'pl...",Inconnu,Inconnu,Inconnu,6.7,1.400,Indisponible,The Love Europe Project is a collection of 9 s...,/7TzHFu7InuyhJ2xUiHfJ07aszTm.jpg,False,0,0,Drama,NaN,NaN
7983,tt9894450,Felicità,2020,81,['fr'],Bruno Merle,"writer,producer,director","tt14485734,tt9894450,tt23898914,tt0814143",6.6,5.208,Indisponible,"Tommy, 11 years old, is on the road again with...",/yhAiRXQUkMRnalRZFr1jiRCohjM.jpg,False,0,236880,Comedy,Drama,NaN


In [3]:
df["overview"]

0       When her scientist ex-boyfriend discovers a po...
1       Steph, Jean-Claude and Jacques work in a Paris...
2       The old Valmorin died 200 years ago. The notar...
3       On crowded Milan streets, two men execute a sp...
4       This drama about the Carmelite order of nuns i...
                              ...                        
7980    Nina and Madeleine have been sharing their liv...
7981    Three officers are tasked with escorting an il...
7982    The Love Europe Project is a collection of 9 s...
7983    Tommy, 11 years old, is on the road again with...
7984    A psychiatric hospital patient pretends to be ...
Name: overview, Length: 7985, dtype: str

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7985 entries, 0 to 7984
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             7985 non-null   str    
 1   title              7985 non-null   str    
 2   startYear          7985 non-null   int64  
 3   runtimeMinutes     7985 non-null   int64  
 4   spoken_languages   7985 non-null   str    
 5   primaryName        7985 non-null   str    
 6   primaryProfession  7985 non-null   str    
 7   knownForTitles     7985 non-null   str    
 8   averageRating      7985 non-null   float64
 9   popularity         7985 non-null   float64
 10  tagline            7985 non-null   str    
 11  overview           7985 non-null   str    
 12  poster_path        7985 non-null   str    
 13  video              7985 non-null   bool   
 14  budget             7985 non-null   int64  
 15  revenue            7985 non-null   int64  
 16  genre_1            7985 non-null   

In [5]:
import nltk
import os

venv_nltk_path = r"J:\Projet GIT\Projet2_Cinema_Creuse\.venv\nltk_data"
nltk.data.path.insert(0, venv_nltk_path)
os.environ["NLTK_DATA"] = venv_nltk_path

# Télécharger les stopwords dans ce dossier
nltk.download('stopwords', download_dir=venv_nltk_path)

[nltk_data] Downloading package stopwords to J:\Projet
[nltk_data]     GIT\Projet2_Cinema_Creuse\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
import re
import nltk
from textblob import TextBlob as tb

def clean_fr(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("french"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

def clean_en(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

In [7]:
from nltk.corpus import stopwords
stop_fr = stopwords.words("french")

In [36]:
X = df[["primaryName", "overview", "tagline", "startYear", "runtimeMinutes","genre_1", "genre_2", "genre_3"]]#"budget", "revenue", "title", 

for col in ['tagline', 'overview']:# "title",
    X[col] = X[col].replace("Inconnu", "")

X["primaryName"] = X["primaryName"].apply(lambda x : x.replace(" ", ""))

#X["titlelema"] = X["title"].apply(clean_fr)
X['tagline'] = X['tagline'].apply(clean_en)
X['overview'] = X['overview'].apply(clean_en)

nfeature = ["startYear",  "runtimeMinutes" ] #"budget", "revenue"
cfeature = ["primaryName","genre_1", "genre_2", "genre_3"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), nfeature),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cfeature),
        #('text_title', TfidfVectorizer(max_features=5000, stop_words=stop_fr), 'titlelema'),
        ('text_tagline', TfidfVectorizer(max_features=5000, stop_words= "english"), 'tagline'),
        ('text_overview', TfidfVectorizer(max_features=5000, stop_words= "english"), 'overview')
    ])

        

In [37]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),                                  # Étape 1 : preprocessing des valeur num (scaler) et valeur cat (encoder)(je ne les scale pas), tfidf sur les textes
    ('model', NearestNeighbors(n_neighbors=3, metric='euclidean'))   # Étape 2 : On donne les données propres au modèle
])
pipeline.fit(X)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [38]:
idx_film = df.index[df["title"] == "Le Fabuleux Destin d'Amélie Poulain"][0]
donnees_film = X.loc[[idx_film]]

film_transformed = pipeline.named_steps['preprocessor'].transform(donnees_film)
distances, indices = pipeline.named_steps['model'].kneighbors(film_transformed, n_neighbors=11)
voisin_indices = indices[0][1:]  # à partir du 1er élément

voisins = df.iloc[voisin_indices]
print(voisins[["title", "genre_1", "genre_2", "genre_3", "startYear"]])

                                       title genre_1  genre_2 genre_3  \
2525                                 Western  Comedy  Romance     NaN   
2447                       Dieu seul me voit  Comedy  Romance     NaN   
3686                            Esprit libre  Comedy  Romance     NaN   
3235                  Le triomphe de l'amour  Comedy  Romance     NaN   
4315                       Comme des voleurs  Comedy  Romance     NaN   
2792                                Kimberly  Comedy  Romance     NaN   
2398                       Portraits chinois  Comedy  Romance     NaN   
3641                           Après vous...  Comedy  Romance     NaN   
6562               Sous les jupes des filles  Comedy  Romance     NaN   
4709  Bancs publics (Versailles rive droite)  Comedy  Romance     NaN   

      startYear  
2525       1997  
2447       1998  
3686       2004  
3235       2001  
4315       2006  
2792       1999  
2398       1996  
3641       2003  
6562       2014  
4709       2009 

In [35]:
old_rec = list(df.iloc[indices[0][1:]]["title"])  # stocker ici
print("OLD REC =", old_rec)

OLD REC = ['Western', 'Dieu seul me voit', 'Esprit libre', "L'arnacoeur", "Le triomphe de l'amour", 'Comme des voleurs', 'Kimberly', 'Portraits chinois', 'Après vous...', 'Sous les jupes des filles']


In [39]:
new_rec = list(df.iloc[indices[0][1:]]["title"])
print("NEW REC =", new_rec)

NEW REC = ['Western', 'Dieu seul me voit', 'Esprit libre', "Le triomphe de l'amour", 'Comme des voleurs', 'Kimberly', 'Portraits chinois', 'Après vous...', 'Sous les jupes des filles', 'Bancs publics (Versailles rive droite)']


In [40]:


old_set = set(old_rec)
new_set = set(new_rec)

jaccard = len(old_set & new_set) / len(old_set | new_set)
print(jaccard)

def jaccard_score(rec1, rec2):
    set1, set2 = set(rec1), set(rec2)
    return len(set1 & set2) / len(set1 | set2)

score = jaccard_score(old_rec, new_rec)
print("Jaccard :", score)

0.8181818181818182
Jaccard : 0.8181818181818182
